# Regime Structure 03 — big-move days vs normal days

Pre-registration: `discussions/Regime_Structure/007_claude_rs03_preregistration.md`. Code: `src/data_engineering/modeling/big_move_days.py`.

- **Session:** the 24 h ending at 08:00 UTC (Deribit daily expiry). Binance USD-M 1m → 5m bars, BTC/ETH/SOL/XRP, about 2 years.
- **Big-move day:** $z=\lvert\Delta\log P_{\text{session}}\rvert/\hat\sigma$ with $\hat\sigma$ = RMS of the previous 7 sessions' realised vol; BIG if $z\ge q_{85}$ (development window, per asset).
- **DURING** (what happens on those days): descriptive only. **PRE** (known at 08:00, target BIG) and **EARLY** (08:00–10:00, target BIG over the *rest* of the session 10:00→08:00): AUC big vs normal, screened on the development window, confirmed on the test window (from 2026-04-01) at a Bonferroni level over the candidates.
- **Clustering:** do big days follow big days, and do they hit all assets at once?

Environment: `RS03_ASSETS`, `RS03_N_BOOT` (default 2000), `RS03_SYNTHETIC=1`.

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import big_move_days as bm
bm = importlib.reload(bm)
from dataclasses import replace

SYNTHETIC = os.getenv('RS03_SYNTHETIC', '').strip() == '1'
ASSETS = [a.strip().upper() for a in os.getenv('RS03_ASSETS', ','.join(bm.ASSETS)).split(',') if a.strip()]
P = replace(bm.Params(), n_boot=int(os.getenv('RS03_N_BOOT', str(bm.Params().n_boot))))
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='rs03_')); BIN_DIR = None
else:
    BIN_DIR = Path(os.getenv('BINANCE_UM_DIR') or Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Regime_Structure' / '03'
OUT.mkdir(parents=True, exist_ok=True)
print('RS 03 |', bm.VERSION, '| synthetic' if SYNTHETIC else f'| data {BIN_DIR}', '| out', OUT)

## 1. Sessions and big-move labels

In [ ]:
RES = {}
for i, a in enumerate(ASSETS):
    if SYNTHETIC:
        df = bm.synthetic_klines(n_days=760, seed=50 + i, start='2024-10-01')
    else:
        path = BIN_DIR / f'{a}USDT_um_1m.parquet'
        if not path.exists():
            print(f'{a}: no Binance file -> skipped'); continue
        df = pd.read_parquet(path, columns=['open_time', 'close', 'volume', 'taker_buy_volume', 'trades'])
    RES[a] = bm.run_asset(df, a, P)
    del df
T = pd.concat([r['table'] for r in RES.values()], ignore_index=True)
cov = T.dropna(subset=['big']).groupby(['asset', 'window']).agg(sessions=('big', 'size'), big_days=('big', 'sum'),
                                                               big_share=('big', 'mean'), z_median=('z', 'median'))
cov['z_threshold'] = [RES[a]['threshold'] for a, _ in cov.index]
display(cov.round(3))
print('A big day here is a session whose open-to-close move is at least', round(np.mean([r['threshold'] for r in RES.values()]), 2),
      'x the 7-session vol forecast (mean over assets).')

## 2. What happens on big days (descriptive, DURING)

In [ ]:
desc = []
for c in bm.DURING:
    for w in ('dev', 'test'):
        d = T[(T.window == w) & T.big.notna()]
        desc.append({'feature': c, 'window': w, 'median_big': d.loc[d.big == 1, c].median(),
                     'median_normal': d.loc[d.big == 0, c].median(), 'auc': bm.auc(d[c].to_numpy(float), d.big.to_numpy(bool))})
desc = pd.DataFrame(desc)
display(desc.pivot_table(index='feature', columns='window', values=['median_big', 'median_normal', 'auc'], sort=False).round(3))
top = (T[T.big == 1].groupby('window')['dur_top_hour'].value_counts(normalize=True).rename('share').reset_index())
top['utc_hour'] = (8 + top['dur_top_hour'].astype(int)) % 24
print('UTC hour holding the largest hourly move on BIG days (top 6, dev):')
display(top[top.window == 'dev'].nlargest(6, 'share')[['utc_hour', 'share']].round(3))

In [ ]:
prof = []
for a, r in RES.items():
    prof.append(bm.hourly_profile(r['H'], r['V'], r['table']).assign(asset=a))
prof = pd.concat(prof).groupby(['set', 'hour_after_0800', 'utc_hour'], as_index=False)[['abs_ret_rel', 'volume_rel']].mean()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for s, col in (('big', '#c0392b'), ('normal', '#7f8c8d')):
    g = prof[prof.set == s]
    ax[0].plot(g.hour_after_0800, g.abs_ret_rel, '-o', ms=3, color=col, label=s)
    ax[1].plot(g.hour_after_0800, g.volume_rel, '-o', ms=3, color=col, label=s)
for k, ttl in enumerate(('|hourly return| / hourly vol forecast', 'hourly volume / 30-session median for that hour')):
    ax[k].set_xticks(range(0, 24, 2), [f'{(8 + h) % 24:02d}' for h in range(0, 24, 2)]); ax[k].set_xlabel('UTC hour')
    ax[k].set_title(ttl, fontsize=10); ax[k].legend()
plt.tight_layout(); plt.show()

## 3. What comes before big days (PRE at 08:00, EARLY 08:00–10:00) — screened on dev, confirmed on test

In [ ]:
pre = bm.screen(T, bm.PRE, P).assign(group='PRE')
early = bm.screen(T, bm.EARLY, P, target='big_rest').assign(group='EARLY')
SCR = pd.concat([pre, early], ignore_index=True)
cols = ['group', 'feature', 'median_big_dev', 'median_normal_dev', 'auc_dev', 'lo_dev', 'hi_dev', 'candidate',
        'auc_test', 'lo_test', 'hi_test', 'test_level', 'confirmed']
display(SCR[cols].round(3))
print('CONFIRMED:', SCR.loc[SCR.confirmed, ['group', 'feature', 'auc_dev', 'auc_test']].round(3).to_dict(orient='records'))

In [ ]:
pa = []
for a in RES:
    d = T[(T.asset == a) & (T.window == 'test') & T.big.notna()]
    for c in bm.PRE + bm.EARLY:
        tg = 'big_rest' if c in bm.EARLY else 'big'
        dd = d[d[tg].notna()]
        pa.append({'asset': a, 'feature': c, 'auc_test': bm.auc(dd[c].to_numpy(float), dd[tg].to_numpy(bool))})
pa = pd.DataFrame(pa).pivot(index='feature', columns='asset', values='auc_test')
print('Per-asset AUC, test window (descriptive):'); display(pa.loc[bm.PRE + bm.EARLY].round(3))

## 4. Do big days cluster? (a regime question)

In [ ]:
CL = bm.clustering(T, P)
display(CL.round(4))
dir_ = T[T.big == 1].groupby('window')['up'].mean()
print('Share of big days that are up-moves:', dir_.round(3).to_dict())

## 5. Save compact outputs

In [ ]:
cov.to_csv(OUT / '03_coverage.csv')
desc.to_csv(OUT / '03_during.csv', index=False)
top.to_csv(OUT / '03_top_hour.csv', index=False)
prof.to_csv(OUT / '03_hourly_profile.csv', index=False)
SCR.to_csv(OUT / '03_screen.csv', index=False)
pa.to_csv(OUT / '03_pre_early_by_asset_test.csv')
CL.to_csv(OUT / '03_clustering.csv', index=False)
manifest = {'registration': bm.registration(P), 'synthetic': SYNTHETIC, 'assets': list(RES),
            'thresholds': {a: r['threshold'] for a, r in RES.items()},
            'confirmed': SCR.loc[SCR.confirmed, 'feature'].tolist(), 'run_utc': str(pd.Timestamp.now(tz='UTC'))}
(OUT / '03_manifest.json').write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
print('saved to', OUT, sorted(p.name for p in OUT.iterdir()))